In [79]:
import pandas as pd
import os
import kagglehub
import zipfile

In [80]:
def run_vietnam_rent_etl():
    print("Extract dataset from Kaggle API...")
    try:
        download_path = kagglehub.dataset_download(
            "cresht2606/vietnam-real-estate-datasets-catalyst")
        print(f"Dataset downloaded to: {download_path}")

        for item in os.listdir(download_path):
            if item.endswith('.zip'):
                zip_ref = zipfile.ZipFile(
                    os.path.join(download_path, item), 'r')
                print(f"Unzipping {item}...")
                zip_ref.extractall(download_path)
                zip_ref.close()
        
        target_file = os.path.join(
            download_path, "house_rental_dec29th_2025.csv")
        
        if not os.path.exists(target_file):
            print(f'Could not find house_rental_dec29th_2025.csv in {download_path}')
            return None
        
        df = pd.read_csv(target_file)
        print(f"Successfully loaded {len(df)} rows.")

        return df
    except Exception as e:
        print(f"Extraction failed: {e}")
        return

In [81]:
if __name__ == "__main__":
    run_vietnam_rent_etl()
    df_raw = run_vietnam_rent_etl()

Extract dataset from Kaggle API...
Dataset downloaded to: C:\Users\Admin\.cache\kagglehub\datasets\cresht2606\vietnam-real-estate-datasets-catalyst\versions\1
Successfully loaded 4808 rows.
Extract dataset from Kaggle API...
Dataset downloaded to: C:\Users\Admin\.cache\kagglehub\datasets\cresht2606\vietnam-real-estate-datasets-catalyst\versions\1
Successfully loaded 4808 rows.


In [82]:
# We can see that the location has both the district and the city. And some have the city and the province.
print(df_raw.head())

       id                                         detail_url  \
0  196787  https://batdongsan.vn/cho-thue-nha-mat-tien-th...   
1  196777  https://batdongsan.vn/thue-nha-nguyen-can-hxt-...   
2  196689  https://batdongsan.vn/linhhk-cho-thue-nha-hoan...   
3  142301  https://batdongsan.vn/cho-thue-toa-nha-2-mat-t...   
4  142012  https://batdongsan.vn/10-trieuthang-cho-thue-n...   

                                               title              location  \
0  Cho thuê nhà Mặt Tiền Thạch Lam 92m², 4TẦNG - ...  Tân Phú, Hồ Chí Minh   
1  Thuê nhà nguyên căn HXT  lô góc Quận 8, 4 tầng...   Quận 8, Hồ Chí Minh   
2  LinhHK cho thuê nhà Hoàn Kiếm – 150m² x 5 tầng...     Hoàn Kiếm, Hà Nội   
3  Cho thuê Tòa nhà 2 Mặt tiền Diệp Minh Châu 160...  Tân Phú, Hồ Chí Minh   
4  🏠 10 triệu/tháng 🏠 CHO THUÊ NHÀ NGUYÊN CĂN 3 T...   Huế, Thừa Thiên Huế   

   timeline_hours  area_m2  bedrooms  bathrooms  floors  frontage  \
0              48     92.0       5.0        5.0     5.0      True   
1       

In [83]:
# We can also see that there are 4808 id's, but the area, bedrooms, bathrooms, floors, price_million_vnd are all having less than that.
print(df_raw.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4808 entries, 0 to 4807
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 4808 non-null   int64  
 1   detail_url         4808 non-null   object 
 2   title              4808 non-null   object 
 3   location           4808 non-null   object 
 4   timeline_hours     4808 non-null   int64  
 5   area_m2            3889 non-null   float64
 6   bedrooms           2981 non-null   float64
 7   bathrooms          3013 non-null   float64
 8   floors             2623 non-null   float64
 9   frontage           4808 non-null   bool   
 10  price_million_vnd  4691 non-null   float64
dtypes: bool(1), float64(5), int64(2), object(3)
memory usage: 380.4+ KB
None


In [84]:
import numpy as np

In [ ]:
def transform_vietnam_rent_data(df_raw):
    print("Transforming...")

    df_clean = df_raw.copy()

    # Drop the rows that have missing important values
    df_clean.dropna(subset=['area_m2', 'price_million_vnd'], inplace=True)
    rows_dropped = len(df_raw) - len(df_clean)
    print(f'{rows_dropped} rows dropped because of missing data in area and price.')

    # Since the number of missing values from bedrooms, bathrooms, floors are too many, and it usually just mean that it's a studio,...
    # So I will replace them with a "1" and as integer, since there are no 1.5 bed room.
    df_clean['bedrooms'] = df_clean['bedrooms'].fillna(1).astype(int)
    df_clean['bathrooms'] = df_clean['bathrooms'].fillna(1).astype(int)
    df_clean['floors'] = df_clean['floors'].fillna(1).astype(int)
    print('Replace the missing bedrooms, bathrooms, and floors with 1.')

    # I will seperate the location into 2, but there are cities inside provinces and district inside cities, I will divide them into
    # 2 levels instead of just cities and districts
    df_clean[['location_level_2', 'location_level_1']] = df_clean['location'].str.split(', ', n=1, expand=True)
    print('Successfully splitted location into 2 parts.')

    # Calculate millions into the raw number, and then divide by area to find the KPI
    df_clean = df_clean[(df_clean['price_million_vnd'] > 0)
                        & (df_clean['area_m2'] > 0)]
    df_clean['price_per_m2_vnd'] = ((df_clean['price_million_vnd']*1000000)/df_clean['area_m2']).round(0)

    # Some property have outlier price, so we also remove that by choosing only from the middle 50% of both
    Q1 = df_clean['price_per_m2_vnd'].quantile(0.25)
    Q3 = df_clean['price_per_m2_vnd'].quantile(0.75)
    IQR = Q3 - Q1
    # After checking, the data seems to be right skewed, so the lower bound calculated by IQR will negative, so I just give a ceiling
    lower_bound = 30000
    upper_bound = Q3 + 1.5 * IQR
    # Keep only the rows inside
    df_clean = df_clean[(df_clean['price_per_m2_vnd'] >= lower_bound) & (
        df_clean['price_per_m2_vnd'] <= upper_bound)]

    # Since this data seems to have some industrial area too, I will put a ceiling for the area too
    # 1000 m2 is a safe cutoff for residential/standard commercial
    df_clean = df_clean[df_clean['area_m2'] <= 1000]

    # Since there are properties that are posted by different people, I will deal with it by removing the ones that have the same
    # location, area, bedrooms, price.
    df_clean = df_clean.drop_duplicates(
        subset=['location_level_2', 'area_m2', 'bedrooms', 'price_million_vnd'], keep='first')
    print(f"Final row count: {len(df_clean)}")

    print(f'Transformation completed. Total of {len(df_clean)} rows ready.')

    return df_clean

In [86]:
if __name__ == "__main__":
    if df_raw is not None:
        df_transformed = transform_vietnam_rent_data(df_raw)
        print("Preview cleaned data...")
        print(df_transformed.head())

Transforming...
921 rows dropped because of missing data in area and price.
Replace the missing bedrooms, bathrooms, and floors with 1.
Successfully splitted location into 2 parts.
Final row count: 2952
Transformation completed. Total of 2952 rows ready.
Preview cleaned data...
       id                                         detail_url  \
0  196787  https://batdongsan.vn/cho-thue-nha-mat-tien-th...   
1  196777  https://batdongsan.vn/thue-nha-nguyen-can-hxt-...   
2  196689  https://batdongsan.vn/linhhk-cho-thue-nha-hoan...   
3  142301  https://batdongsan.vn/cho-thue-toa-nha-2-mat-t...   
4  142012  https://batdongsan.vn/10-trieuthang-cho-thue-n...   

                                               title              location  \
0  Cho thuê nhà Mặt Tiền Thạch Lam 92m², 4TẦNG - ...  Tân Phú, Hồ Chí Minh   
1  Thuê nhà nguyên căn HXT  lô góc Quận 8, 4 tầng...   Quận 8, Hồ Chí Minh   
2  LinhHK cho thuê nhà Hoàn Kiếm – 150m² x 5 tầng...     Hoàn Kiếm, Hà Nội   
3  Cho thuê Tòa nhà 2 Mặ

In [87]:
import sqlite3

In [88]:
def load_vietnam_rent_data(df_clean, db_name="vietnam_rentals.db"):
    print("Loading data...")

    # Create the db file
    # conn = sqlite3.connect(db_name)
    # # Load data to table 'listings'
    # df_clean.to_sql('listings', conn, if_exists='replace', index=False)
    # conn.close()

    df_clean.to_csv('vietnam_rentals_powerbi.csv', index=False)
    print(f"Data successfully saved as csv.")

In [89]:
if __name__ == "__main__":
    # Extract
    df_raw = run_vietnam_rent_etl()

    if df_raw is not None:
        # Transform
        df_clean = transform_vietnam_rent_data(df_raw)

        # Load
        load_vietnam_rent_data(df_clean)

Extract dataset from Kaggle API...
Dataset downloaded to: C:\Users\Admin\.cache\kagglehub\datasets\cresht2606\vietnam-real-estate-datasets-catalyst\versions\1
Successfully loaded 4808 rows.
Transforming...
921 rows dropped because of missing data in area and price.
Replace the missing bedrooms, bathrooms, and floors with 1.
Successfully splitted location into 2 parts.
Final row count: 2952
Transformation completed. Total of 2952 rows ready.
Loading data...
Data successfully saved as csv.
